# World Bank Data Collection and Cleaning

## Objective

This notebook loads and cleans World Bank World Development Indicators data for the EU-27 countries.

The raw dataset contains annual observations from 1995 to 2025. The cleaned output will be used later for exploratory analysis, feature engineering and GDP-growth forecasting.

This notebook does not build machine-learning models yet.

## 1. Import libraries

We use:

- `pandas` for reading and transforming data.
- `numpy` for numerical operations.
- `pathlib` for working with file paths.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

## 2. Define file paths

The raw dataset is stored in the `data/raw/` folder.

We will keep the raw file unchanged and create a separate cleaned output.

In [4]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

raw_path = project_root / "data" / "raw" / "world_bank_wdi_eu27_1995_2025.csv"
processed_folder = project_root / "data" / "processed"

processed_folder.mkdir(parents=True, exist_ok=True)

print("Raw file:", raw_path)
print("Raw file exists:", raw_path.exists())

Raw file: /Users/sozdemirdeni/ML-Project/data/raw/world_bank_wdi_eu27_1995_2025.csv
Raw file exists: True


## 3. Load the raw dataset

The downloaded World Bank file is in a wide format:

- Each row represents one country and indicator.
- Each year is a separate column.

We will inspect the file before cleaning it.

In [5]:
raw_data = pd.read_csv(raw_path)

print("Rows:", raw_data.shape[0])
print("Columns:", raw_data.shape[1])

raw_data.head()

Rows: 275
Columns: 35


,Country Name,Country Code,Series Name,Series Code,1995 [YR1995],1996 [YR1996],1997 [YR1997],1998 [YR1998],1999 [YR1999],2000 [YR2000],...,2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
0,Austria,AUT,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,2.24336631500898,1.86097115760437,1.30597857227691,0.922467196680849,0.568993765885531,2.34486285357281,...,0.891592,2.081269,1.998380,1.530896,1.381911,2.766667,8.546870,7.814134,2.937916,3.52719439956928
1,Austria,AUT,Population growth (annual %),SP.POP.GROW,0.153106260702748,0.135019833748789,0.113316608322994,0.109728368100981,0.194563153264961,0.240466652446524,...,1.081396,0.694621,0.487072,0.444674,0.415177,0.435672,0.956288,0.989465,0.504880,0.328301850203979
2,Austria,AUT,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,2.66798388882503,2.21576713087359,2.1458350552646,3.49288002962309,3.76192435190464,3.18952292988787,...,2.117220,2.272250,2.484221,1.754976,-6.318255,4.923092,5.330964,-0.786244,-0.659090,0.615747280959056
3,Austria,AUT,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,33603.4278263558,34301.6562151382,34998.0321173934,36180.7490571265,37468.8698869794,38571.0858780001,...,44362.670281,45056.638591,45951.581916,46550.562246,43428.698314,45368.643434,47332.424036,46497.911464,45958.825007,46090.2514170459
4,Austria,AUT,Gross fixed capital formation (% of GDP),NE.GDI.FTOT.ZS,25.6814352868846,25.9792394170832,25.5410735111136,25.5828501885423,25.0791471023159,25.706049968312,...,23.321910,23.845945,24.313594,25.080112,25.134257,25.864654,25.212720,24.687558,23.547911,23.5267100229664


## 4. Inspect the raw data

Before cleaning, we check:

- Column names
- Data types
- Missing values
- Metadata rows included in the download

In [6]:
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275 entries, 0 to 274
Data columns (total 35 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Country Name   272 non-null    object 
 1   Country Code   270 non-null    object 
 2   Series Name    270 non-null    object 
 3   Series Code    270 non-null    object 
 4   1995 [YR1995]  270 non-null    object 
 5   1996 [YR1996]  270 non-null    object 
 6   1997 [YR1997]  270 non-null    object 
 7   1998 [YR1998]  270 non-null    object 
 8   1999 [YR1999]  270 non-null    object 
 9   2000 [YR2000]  270 non-null    object 
 10  2001 [YR2001]  270 non-null    object 
 11  2002 [YR2002]  270 non-null    float64
 12  2003 [YR2003]  270 non-null    float64
 13  2004 [YR2004]  270 non-null    float64
 14  2005 [YR2005]  270 non-null    float64
 15  2006 [YR2006]  270 non-null    float64
 16  2007 [YR2007]  270 non-null    float64
 17  2008 [YR2008]  270 non-null    float64
 18  2009 [YR20

In [7]:
raw_data.tail()

,Country Name,Country Code,Series Name,Series Code,1995 [YR1995],1996 [YR1996],1997 [YR1997],1998 [YR1998],1999 [YR1999],2000 [YR2000],...,2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
270,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
271,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
273,Data from database: World Development Indicators,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
274,Last Updated: 07/13/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Remove metadata and footer rows

Valid data rows contain both:

- A country code
- A series code

Rows without these values are metadata or blank rows.

In [9]:
data = raw_data[
    raw_data["Country Code"].notna()
    & raw_data["Series Code"].notna()
    & (raw_data["Country Code"] != "")
    & (raw_data["Series Code"] != "")
].copy()

print("Rows after removing metadata:", data.shape[0])

Rows after removing metadata: 270


## 6. Confirm the indicator selection

The dataset should contain the target indicator and the proposed predictors.

In [10]:
series_summary = (
    data[["Series Code", "Series Name"]]
    .drop_duplicates()
    .sort_values("Series Code")
)

series_summary

,Series Code,Series Name
6,BX.KLT.DINV.WD.GD.ZS,"Foreign direct investment, net inflows (% of GDP)"
0,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)"
9,NE.CON.GOVT.ZS,General government final consumption expenditu...
7,NE.EXP.GNFS.ZS,Exports of goods and services (% of GDP)
4,NE.GDI.FTOT.ZS,Gross fixed capital formation (% of GDP)
8,NE.IMP.GNFS.ZS,Imports of goods and services (% of GDP)
2,NY.GDP.MKTP.KD.ZG,GDP growth (annual %)
3,NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$)
5,SL.UEM.TOTL.ZS,"Unemployment, total (% of total labor force) (..."
1,SP.POP.GROW,Population growth (annual %)


In [11]:
expected_series = {
    "NY.GDP.MKTP.KD.ZG",
    "NY.GDP.PCAP.KD",
    "FP.CPI.TOTL.ZG",
    "NE.GDI.FTOT.ZS",
    "SL.UEM.TOTL.ZS",
    "NE.EXP.GNFS.ZS",
    "NE.IMP.GNFS.ZS",
    "BX.KLT.DINV.WD.GD.ZS",
    "NE.CON.GOVT.ZS",
    "SP.POP.GROW",
}

actual_series = set(data["Series Code"].unique())

print("Missing expected indicators:", expected_series - actual_series)
print("Unexpected indicators:", actual_series - expected_series)

Missing expected indicators: set()
Unexpected indicators: set()


## 7. Keep only the EU-27 countries

The analysis focuses on the 27 current EU member states.

In [12]:
eu27_codes = [
    "AUT", "BEL", "BGR", "HRV", "CYP", "CZE", "DNK",
    "EST", "FIN", "FRA", "DEU", "GRC", "HUN", "IRL",
    "ITA", "LVA", "LTU", "LUX", "MLT", "NLD", "POL",
    "PRT", "ROU", "SVK", "SVN", "ESP", "SWE"
]

data = data[data["Country Code"].isin(eu27_codes)].copy()

print("Number of countries:", data["Country Code"].nunique())
print(sorted(data["Country Code"].unique()))

Number of countries: 27
['AUT', 'BEL', 'BGR', 'CYP', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'LUX', 'LVA', 'MLT', 'NLD', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'SWE']


## 8. Reshape the data from wide to long format

The raw data has one column for every year.

We will convert it into long format so that every row represents:

```text
One country + one indicator + one year

In [15]:
year_columns = [
    column for column in data.columns
    if column[:4].isdigit()
]

id_columns = [
    "Country Name",
    "Country Code",
    "Series Name",
    "Series Code"
]

long_data = data.melt(
    id_vars=id_columns,
    value_vars=year_columns,
    var_name="Year",
    value_name="Value"
)

long_data.head()

,Country Name,Country Code,Series Name,Series Code,Year,Value
0,Austria,AUT,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,1995 [YR1995],2.24336631500898
1,Austria,AUT,Population growth (annual %),SP.POP.GROW,1995 [YR1995],0.153106260702748
2,Austria,AUT,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,1995 [YR1995],2.66798388882503
3,Austria,AUT,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,1995 [YR1995],33603.4278263558
4,Austria,AUT,Gross fixed capital formation (% of GDP),NE.GDI.FTOT.ZS,1995 [YR1995],25.6814352868846


In [16]:
long_data["Year"] = (
    long_data["Year"]
    .str.extract(r"(\d{4})")[0]
    .astype(int)
)

long_data["Value"] = pd.to_numeric(
    long_data["Value"],
    errors="coerce"
)

long_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8370 entries, 0 to 8369
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Country Name  8370 non-null   object 
 1   Country Code  8370 non-null   object 
 2   Series Name   8370 non-null   object 
 3   Series Code   8370 non-null   object 
 4   Year          8370 non-null   int64  
 5   Value         8356 non-null   float64
dtypes: float64(1), int64(1), object(4)
memory usage: 392.5+ KB


## 9. Check duplicate observations

Each country, year and indicator combination should appear only once.

In [17]:
duplicate_count = long_data.duplicated(
    subset=["Country Code", "Year", "Series Code"]
).sum()

print("Duplicate observations:", duplicate_count)

Duplicate observations: 0


## 10. Pivot indicators into columns

The long dataset currently has one row per country, year and indicator.

We will pivot it so that each indicator becomes a separate column.

In [18]:
clean_data = (
    long_data
    .pivot_table(
        index=["Country Code", "Country Name", "Year"],
        columns="Series Code",
        values="Value",
        aggfunc="first"
    )
    .reset_index()
)

clean_data.head()

Series Code,Country Code,Country Name,Year,BX.KLT.DINV.WD.GD.ZS,FP.CPI.TOTL.ZG,NE.CON.GOVT.ZS,NE.EXP.GNFS.ZS,NE.GDI.FTOT.ZS,NE.IMP.GNFS.ZS,NY.GDP.MKTP.KD.ZG,NY.GDP.PCAP.KD,SL.UEM.TOTL.ZS,SP.POP.GROW
0,AUT,Austria,1995,0.760156,2.243366,19.913675,33.665001,25.681435,34.860005,2.667984,33603.427826,4.345,0.153106
1,AUT,Austria,1996,1.832365,1.860971,19.919978,34.437736,25.979239,36.031581,2.215767,34301.656215,5.282,0.135020
2,AUT,Austria,1997,1.260587,1.305979,19.907484,37.214434,25.541074,38.030042,2.145835,34998.032117,5.150,0.113317
3,AUT,Austria,1998,2.106185,0.922467,19.880590,38.561591,25.582850,38.787931,3.492880,36180.749057,5.483,0.109728
4,AUT,Austria,1999,1.374399,0.568994,20.132107,39.509313,25.079147,39.053620,3.761924,37468.869887,4.699,0.194563


## 11. Rename indicator columns

The World Bank codes are useful for reproducibility, but descriptive names make the analysis easier to read.

In [19]:
rename_columns = {
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",
    "NY.GDP.PCAP.KD": "gdp_per_capita",
    "FP.CPI.TOTL.ZG": "inflation",
    "NE.GDI.FTOT.ZS": "gross_fixed_capital_formation",
    "SL.UEM.TOTL.ZS": "unemployment",
    "NE.EXP.GNFS.ZS": "exports_percent_gdp",
    "NE.IMP.GNFS.ZS": "imports_percent_gdp",
    "BX.KLT.DINV.WD.GD.ZS": "fdi_inflows_percent_gdp",
    "NE.CON.GOVT.ZS": "government_consumption_percent_gdp",
    "SP.POP.GROW": "population_growth"
}

clean_data = clean_data.rename(columns=rename_columns)

clean_data.head()

Series Code,Country Code,Country Name,Year,fdi_inflows_percent_gdp,inflation,government_consumption_percent_gdp,exports_percent_gdp,gross_fixed_capital_formation,imports_percent_gdp,gdp_growth,gdp_per_capita,unemployment,population_growth
0,AUT,Austria,1995,0.760156,2.243366,19.913675,33.665001,25.681435,34.860005,2.667984,33603.427826,4.345,0.153106
1,AUT,Austria,1996,1.832365,1.860971,19.919978,34.437736,25.979239,36.031581,2.215767,34301.656215,5.282,0.135020
2,AUT,Austria,1997,1.260587,1.305979,19.907484,37.214434,25.541074,38.030042,2.145835,34998.032117,5.150,0.113317
3,AUT,Austria,1998,2.106185,0.922467,19.880590,38.561591,25.582850,38.787931,3.492880,36180.749057,5.483,0.109728
4,AUT,Austria,1999,1.374399,0.568994,20.132107,39.509313,25.079147,39.053620,3.761924,37468.869887,4.699,0.194563


## 12. Create trade openness

Trade openness will be calculated as:

```text
Exports (% of GDP) + Imports (% of GDP)

In [20]:
clean_data["trade_openness"] = (
    clean_data["exports_percent_gdp"]
    + clean_data["imports_percent_gdp"]
)

clean_data[
    [
        "Country Code",
        "Year",
        "exports_percent_gdp",
        "imports_percent_gdp",
        "trade_openness"
    ]
].head()

Series Code,Country Code,Year,exports_percent_gdp,imports_percent_gdp,trade_openness
0,AUT,1995,33.665001,34.860005,68.525006
1,AUT,1996,34.437736,36.031581,70.469316
2,AUT,1997,37.214434,38.030042,75.244476
3,AUT,1998,38.561591,38.787931,77.349522
4,AUT,1999,39.509313,39.053620,78.562933


## 13. Check missing values

The FDI indicator has missing historical values for Luxembourg.

We will not impute values in this notebook yet. We will document the missing values and decide how to handle them after creating the modelling split.

In [21]:
missing_summary = (
    clean_data
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary

Series Code
fdi_inflows_percent_gdp               14
Country Code                           0
Country Name                           0
Year                                   0
inflation                              0
government_consumption_percent_gdp     0
exports_percent_gdp                    0
gross_fixed_capital_formation          0
imports_percent_gdp                    0
gdp_growth                             0
gdp_per_capita                         0
unemployment                           0
population_growth                      0
trade_openness                         0
dtype: int64

In [22]:
clean_data[
    clean_data["fdi_inflows_percent_gdp"].isna()
][
    ["Country Code", "Country Name", "Year", "fdi_inflows_percent_gdp"]
]

Series Code,Country Code,Country Name,Year,fdi_inflows_percent_gdp
216,DNK,Denmark,2025,NaN
371,GRC,Greece,2025,NaN
402,HRV,Croatia,2025,NaN
464,IRL,Ireland,2025,NaN
527,LUX,Luxembourg,1995,NaN
528,LUX,Luxembourg,1996,NaN
529,LUX,Luxembourg,1997,NaN
530,LUX,Luxembourg,1998,NaN
531,LUX,Luxembourg,1999,NaN
532,LUX,Luxembourg,2000,NaN


## 14. Sort and validate the cleaned data

The data should be sorted by country and year before feature engineering.

In [24]:
clean_data = (
    clean_data
    .sort_values(["Country Code", "Year"])
    .reset_index(drop=True)
)

print("Rows:", clean_data.shape[0])
print("Countries:", clean_data["Country Code"].nunique())
print("Minimum year:", clean_data["Year"].min())
print("Maximum year:", clean_data["Year"].max())

Rows: 837
Countries: 27
Minimum year: 1995
Maximum year: 2025


In [25]:
expected_columns = [
    "Country Code",
    "Country Name",
    "Year",
    "gdp_growth",
    "gdp_per_capita",
    "inflation",
    "gross_fixed_capital_formation",
    "unemployment",
    "exports_percent_gdp",
    "imports_percent_gdp",
    "fdi_inflows_percent_gdp",
    "government_consumption_percent_gdp",
    "population_growth",
    "trade_openness"
]

missing_columns = [
    column for column in expected_columns
    if column not in clean_data.columns
]

print("Missing expected columns:", missing_columns)

Missing expected columns: []


## 15. Save the cleaned dataset

This file will be used in the next notebook for exploratory analysis and feature engineering.

The raw World Bank file remains unchanged.

In [26]:
clean_path = (
    processed_folder
    / "clean_wdi_eu27_1995_2025.csv"
)

clean_data.to_csv(clean_path, index=False)

print("Saved cleaned dataset to:")
print(clean_path)

Saved cleaned dataset to:
/Users/sozdemirdeni/ML-Project/data/processed/clean_wdi_eu27_1995_2025.csv


In [27]:
assert clean_data["Country Code"].nunique() == 27
assert clean_data["Year"].min() == 1995
assert clean_data["Year"].max() == 2025
assert clean_data.shape[0] == 837
assert duplicate_count == 0

print("All final validation checks passed.")

All final validation checks passed.
